## Behaviour if grouping and calculating the mean if NaN-values are in the group

In [1]:
import pandas as pd
import numpy as np
import holidays
holidays_de= holidays.Germany()

Extra code run to make holidays run:
import sys
!{sys.executable} -m pip install --force-reinstall holidays
!{sys.executable} -m pip show holidays


In [2]:
# Load merged hourly dataset
df = pd.read_csv("../../data_cleaned/merged/Data_imputed_2019_to_2025.csv")
# Convert timestamp columns to datetime
df["period_start_utc"] = pd.to_datetime(df["period_start_utc"], utc=True, errors="coerce")
df["period_end_utc"] = pd.to_datetime(df["period_end_utc"], utc=True, errors="coerce")
df["date"] = pd.to_datetime(df["date"], errors="coerce")

df = df.sort_values("period_start_utc")

df.head()

,date,year,month,day,dayofyear,hour,week,dayofweek,price,period_start_utc,...,on_wind_da,on_wind_act,solar_da,solar_act,gen_forecast_da,gen_actual,res_sum_da,res_sum_act,imputed,interpolated
0,2015-01-04,2015,1,4,4,23,1,6,22.34,2015-01-04 23:00:00+00:00,...,11675.5250,14223.2775,0.0,0.1700,NaN,57028.5925,11867.6500,14701.4675,0,0
1,2015-01-05,2015,1,5,5,0,2,0,17.93,2015-01-05 00:00:00+00:00,...,11924.5575,14207.5025,0.0,0.2075,NaN,56318.8525,12116.9325,14676.2625,0,0
2,2015-01-05,2015,1,5,5,1,2,0,15.17,2015-01-05 01:00:00+00:00,...,12000.4075,14439.8025,0.0,0.1800,NaN,56216.6000,12195.4075,14905.5375,0,0
3,2015-01-05,2015,1,5,5,2,2,0,16.38,2015-01-05 02:00:00+00:00,...,12108.2650,14584.6825,0.0,0.2000,NaN,56394.7350,12301.8900,15038.9000,0,0
4,2015-01-05,2015,1,5,5,3,2,0,17.38,2015-01-05 03:00:00+00:00,...,12196.1175,15071.2450,0.0,0.1950,NaN,57670.7700,12383.7425,15528.9150,0,0


In [3]:

def add_fourier(df, col, period, K=1, drop=False, offset=0):
    # offset=1 is useful for 1-based columns like month/dayofyear/week
    x = df[col].astype(float) - offset
    for k in range(1, K + 1):
        df[f"{col}_sin{k}"] = np.sin(2 * np.pi * k * x / period)
        df[f"{col}_cos{k}"] = np.cos(2 * np.pi * k * x / period)
    if drop:
        df.drop(columns=[col], inplace=True)
    return df

# Example periods
# hour: 0-23 -> P=24, offset=0
# dayofweek: 0-6 or 1-7 -> P=7 (set offset accordingly)
# month: 1-12 -> P=12, offset=1
# dayofyear: 1-365/366 -> usually P=365.25, offset=1
# week: 1-52/53 -> P=52.18 (approx), offset=1

#df = add_fourier(df, "year", 24, K=1, offset=0)
df = add_fourier(df, "dayofyear", 365.25, K=1, offset=1)
df = add_fourier(df, "hour", 24, K=1, offset=0)

df = add_fourier(df, "dayofweek", 7, K=1, offset=0)   # change offset if 1..7
#df = add_fourier(df, "month", 12, K=1, offset=1)
#df = add_fourier(df, "week", 52.18, K=1, offset=1)


In [4]:
df.columns

Index(['date', 'year', 'month', 'day', 'dayofyear', 'hour', 'week',
       'dayofweek', 'price', 'period_start_utc', 'period_end_utc', 'c_by_hour',
       'load_forecast_da', 'load_actual', 'off_wind_da', 'off_wind_act',
       'on_wind_da', 'on_wind_act', 'solar_da', 'solar_act', 'gen_forecast_da',
       'gen_actual', 'res_sum_da', 'res_sum_act', 'imputed', 'interpolated',
       'dayofyear_sin1', 'dayofyear_cos1', 'hour_sin1', 'hour_cos1',
       'dayofweek_sin1', 'dayofweek_cos1'],
      dtype='object')

In [5]:
df.head(10)

,date,year,month,day,dayofyear,hour,week,dayofweek,price,period_start_utc,...,res_sum_da,res_sum_act,imputed,interpolated,dayofyear_sin1,dayofyear_cos1,hour_sin1,hour_cos1,dayofweek_sin1,dayofweek_cos1
0,2015-01-04,2015,1,4,4,23,1,6,22.34,2015-01-04 23:00:00+00:00,...,11867.6500,14701.4675,0,0,0.051584,0.998669,-0.258819,9.659258e-01,-0.781831,0.62349
1,2015-01-05,2015,1,5,5,0,2,0,17.93,2015-01-05 00:00:00+00:00,...,12116.9325,14676.2625,0,0,0.068755,0.997634,0.000000,1.000000e+00,0.000000,1.00000
2,2015-01-05,2015,1,5,5,1,2,0,15.17,2015-01-05 01:00:00+00:00,...,12195.4075,14905.5375,0,0,0.068755,0.997634,0.258819,9.659258e-01,0.000000,1.00000
3,2015-01-05,2015,1,5,5,2,2,0,16.38,2015-01-05 02:00:00+00:00,...,12301.8900,15038.9000,0,0,0.068755,0.997634,0.500000,8.660254e-01,0.000000,1.00000
4,2015-01-05,2015,1,5,5,3,2,0,17.38,2015-01-05 03:00:00+00:00,...,12383.7425,15528.9150,0,0,0.068755,0.997634,0.707107,7.071068e-01,0.000000,1.00000
5,2015-01-05,2015,1,5,5,4,2,0,16.38,2015-01-05 04:00:00+00:00,...,12375.8625,16633.5125,0,0,0.068755,0.997634,0.866025,5.000000e-01,0.000000,1.00000
6,2015-01-05,2015,1,5,5,5,2,0,26.61,2015-01-05 05:00:00+00:00,...,12105.1775,17160.2850,0,0,0.068755,0.997634,0.965926,2.588190e-01,0.000000,1.00000
7,2015-01-05,2015,1,5,5,6,2,0,37.63,2015-01-05 06:00:00+00:00,...,11642.6875,17238.3600,0,0,0.068755,0.997634,1.000000,6.123234e-17,0.000000,1.00000
8,2015-01-05,2015,1,5,5,7,2,0,39.09,2015-01-05 07:00:00+00:00,...,11212.1600,17131.9000,0,0,0.068755,0.997634,0.965926,-2.588190e-01,0.000000,1.00000
9,2015-01-05,2015,1,5,5,8,2,0,41.51,2015-01-05 08:00:00+00:00,...,11672.2400,17691.0500,0,0,0.068755,0.997634,0.866025,-5.000000e-01,0.000000,1.00000


In [6]:
df['is_holiday']= df ['date'].apply(lambda x: 1 if x in holidays_de else 0)

In [7]:
df['day_type']= 'weekday'
df.loc[df['dayofweek'].isin([5,6]), 'day_type'] = 'weekend'
df.loc[df['is_holiday'] == 1, 'day_type'] = 'holiday'
df.head()

,date,year,month,day,dayofyear,hour,week,dayofweek,price,period_start_utc,...,imputed,interpolated,dayofyear_sin1,dayofyear_cos1,hour_sin1,hour_cos1,dayofweek_sin1,dayofweek_cos1,is_holiday,day_type
0,2015-01-04,2015,1,4,4,23,1,6,22.34,2015-01-04 23:00:00+00:00,...,0,0,0.051584,0.998669,-0.258819,0.965926,-0.781831,0.62349,0,weekend
1,2015-01-05,2015,1,5,5,0,2,0,17.93,2015-01-05 00:00:00+00:00,...,0,0,0.068755,0.997634,0.000000,1.000000,0.000000,1.00000,0,weekday
2,2015-01-05,2015,1,5,5,1,2,0,15.17,2015-01-05 01:00:00+00:00,...,0,0,0.068755,0.997634,0.258819,0.965926,0.000000,1.00000,0,weekday
3,2015-01-05,2015,1,5,5,2,2,0,16.38,2015-01-05 02:00:00+00:00,...,0,0,0.068755,0.997634,0.500000,0.866025,0.000000,1.00000,0,weekday
4,2015-01-05,2015,1,5,5,3,2,0,17.38,2015-01-05 03:00:00+00:00,...,0,0,0.068755,0.997634,0.707107,0.707107,0.000000,1.00000,0,weekday


In [8]:
df_2018_2025 = df.query('year >2018')

In [9]:
df['day_type'].value_counts()

day_type
weekday    66815
weekend    27145
holiday     2376
Name: count, dtype: int64

In [10]:
df.shape

(96336, 34)

In [11]:
df_2018_2025.shape

(61367, 34)

In [12]:
df_2018_2025.to_csv("../../data_cleaned/merged/Data_imputed_2019_to_2025_with_Frourier_and_holidays.csv", index=False)

In [13]:
# Optional feature relevance check placeholder
# from sklearn.feature_selection import mutual_info_regression
# mutual_info_regression(df[["res_sum_da"]].shift(1).dropna(), df.price.loc[df[["res_sum_da"]].shift(1).dropna().index])
